## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#include <bits/stdc++.h>

using namespace std;

const int LIM = 1024 + 5;

int N, X, Y;
int blockDim;
int A[LIM], pos[LIM];
vector<int> res;

void refresh() {
    for (int i = 0; i < N; i++) {
        pos[A[i]] = i;
    }
}

void opSwap() {
    res.push_back(0);
    for (int i = 0; i < N; i++) {
        if (A[i] == X) {
            A[i] = Y;
        } else if (A[i] == Y) {
            A[i] = X;
        }
    }
    refresh();
}

void opAdd(int v) {
    v %= N;
    if (v < 0) v += N;
    if (v == 0) return;

    res.push_back(v);
    for (int i = 0; i < N; i++) {
        A[i] = (A[i] + v) % N;
    }
    refresh();
}

void opXor(int v) {
    if (v == 0) return;

    res.push_back(-v);
    for (int i = 0; i < N; i++) {
        A[i] ^= v;
    }
    refresh();
}

struct PermSmall {
    int sz;
    int val[LIM];
    vector<int> ops;

    bool build() {
        vector<int> seen(sz, 0);
        for (int i = 0; i < sz; i++) {
            if (val[i] < 0 || val[i] >= sz) return false;
            seen[val[i]] = 1;
        }
        for (int i = 0; i < sz; i++) {
            if (!seen[i]) return false;
        }

        if (sz == 1) return true;

        PermSmall L, R;
        L.sz = R.sz = sz / 2;
        for (int i = 0; i < sz / 2; i++) {
            L.val[i] = val[i * 2] / 2;
            R.val[i] = val[i * 2 + 1] / 2;
        }

        if (!L.build() || !R.build()) return false;

        if (val[0] & 1) {
            ops.push_back(sz == 2 ? 1 : -1);
        }

        int xL = 0;
        for (int op : L.ops) {
            if (op > 0) {
                ops.push_back(-1);
                ops.push_back(1);
            } else {
                ops.push_back(op * 2);
                xL ^= (-op) * 2;
            }
        }
        if (xL) {
            ops.push_back(-xL);
        }

        int xR = 0;
        for (int op : R.ops) {
            if (op > 0) {
                ops.push_back(1);
                ops.push_back(-1);
            } else {
                ops.push_back(op * 2);
                xR ^= (-op) * 2;
            }
        }

        if ((xL & (sz / 2)) != (xR & (sz / 2))) {
            return false;
        }
        if (xL >= sz / 2) {
            xL -= sz / 2;
        }
        if (xR >= sz / 2) {
            xR -= sz / 2;
        }
        if (xL != xR) {
            return false;
        }

        vector<int> merged;
        for (int op : ops) {
            if (!merged.empty() && op < 0 && merged.back() < 0) {
                merged.back() = -((-merged.back()) ^ (-op));
                if (merged.back() == 0) {
                    merged.pop_back();
                }
            } else {
                merged.push_back(op);
            }
        }
        ops.swap(merged);
        return true;
    }
};

void getMapPair(int u, int v, int &mu, int &mv) {
    int delta = (v - u - blockDim) % N;
    if (delta < 0) delta += N;

    mu = mv = 0;
    for (int step = N / 2; step >= 2 * blockDim; step >>= 1) {
        if (delta >= step) {
            delta -= step;
            mv += step / 2;
        } else {
            mu += step / 2;
        }
    }

    int low = u & (blockDim - 1);
    mu += N / 2 + low;
    mv += low;
}

void swapVal(int u, int v) {
    if (((u / blockDim) & 1) == ((v / blockDim) & 1)) {
        int mid;
        if (((u / blockDim) & 1) == 0) {
            mid = (u & (blockDim - 1)) + blockDim;
        } else {
            mid = u & (blockDim - 1);
        }

        swapVal(u, mid);
        swapVal(v, mid);
        swapVal(u, mid);
        return;
    }

    int mu, mv, m1, m2;
    getMapPair(X, Y, mu, mv);
    getMapPair(u, v, m1, m2);

    opAdd(m1 - u);
    opXor(m1 ^ mu);
    opAdd(X - mu);

    opSwap();

    opAdd(mu - X);
    opXor(m1 ^ mu);
    opAdd(u - m1);
}

void solve() {
    cin >> N >> X >> Y;
    for (int i = 0; i < N; i++) cin >> A[i];
    refresh();

    blockDim = (X - Y + N) % N;
    blockDim &= -blockDim;
    if (blockDim == 0) blockDim = N;

    if (blockDim > 1) {
        PermSmall base;
        base.sz = blockDim;
        for (int i = 0; i < blockDim; i++) {
            base.val[i] = A[i] & (blockDim - 1);
        }

        if (!base.build()) {
            cout << -1 << '\n';
            return;
        }

        for (int op : base.ops) {
            if (op > 0) {
                opAdd(op);
            } else {
                opXor(-op);
            }
        }
    }

    for (int r = 0; r < blockDim; r++) {
        vector<int> vals;
        for (int i = r; i < N; i += blockDim) {
            vals.push_back(A[i]);
        }
        sort(vals.begin(), vals.end());

        for (int i = r, j = 0; i < N; i += blockDim, j++) {
            if (vals[j] != i) {
                cout << -1 << '\n';
                return;
            }
        }

        for (int i = r; i < N; i += blockDim) {
            if (A[i] != i) {
                swapVal(i, A[i]);
            }
        }
    }

    for (int i = 0; i < N; i++) {
        if (A[i] != i) {
            cout << -1 << '\n';
            return;
        }
    }
    if ((int)res.size() > 32768) {
        cout << -1 << '\n';
        return;
    }

    cout << res.size() << '\n';
    for (int op : res) {
        if (op == 0) {
            cout << "0\n";
        } else if (op < 0) {
            cout << "1 " << -op << '\n';
        } else {
            cout << "2 " << op << '\n';
        }
    }
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    solve();
    return 0;
}

## B 长跑

In [ ]:
## add your code here
#include<iostream>
#include<cstdio>
#include<cstring>
#include<cmath>
#include<algorithm>
#include<queue>
using namespace std;
const int MAX_POS = 20005;
struct SupplyPoint {
    int position;
    int cost;
} supply[MAX_POS];
struct State {
    int curPos;
    int remainMoney;
    int idx;
} startState;
bool compareByPos(const SupplyPoint &u, const SupplyPoint &v) {
    if(u.position != v.position) 
        return u.position < v.position;
    return u.cost < v.cost;
}
int main() {
    int pointNum, totalLen, maxEnergy, initMoney;
    while(scanf("%d%d%d%d", &pointNum, &totalLen, &maxEnergy, &initMoney) != EOF) {
        for(int i = 1; i <= pointNum; ++i) {
            scanf("%d%d", &supply[i].position, &supply[i].cost);
        }
        if(maxEnergy >= totalLen) {
            puts("Yes");
            continue;
        }
        ++pointNum;
        supply[pointNum].position = totalLen;
        supply[pointNum].cost = 0;
        sort(supply + 1, supply + pointNum + 1, compareByPos);
        queue<State> stateQueue;
        startState = {0, initMoney, 0};
        stateQueue.push(startState);
        bool isSuccess = false;
        while(!stateQueue.empty()) {
            State curState = stateQueue.front();
            stateQueue.pop();
            if(curState.curPos == totalLen) {
                isSuccess = true;
                break;
            }
            for(int i = curState.idx + 1; i <= pointNum; ++i) {
                int distance = supply[i].position - curState.curPos;
                if(distance > maxEnergy) {
                    break;
                }
                else if(curState.remainMoney >= supply[i].cost) {
                    State nextState = {supply[i].position, initMoney - supply[i].cost, i};
                    stateQueue.push(nextState);
                }
            }
        }
         
        puts(isSuccess ? "Yes" : "No");
    }
     
    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;
using ULL = unsigned long long;
static const ULL SEED = 13331ULL;
string buildString(const string &s, vector<int> &rad) {
    string t = "$#";
    for (char c : s) {
        t += c;
        t += '#';
    }

    int sz = (int)t.size();
    rad.assign(sz, 0);

    int right = 0, center = 0;
    for (int i = 1; i < sz; i++) {
        rad[i] = (right > i) ? min(rad[2 * center - i], right - i) : 1;
        while (i + rad[i] < sz && i - rad[i] >= 0 && t[i + rad[i]] == t[i - rad[i]]) {
            rad[i]++;
        }
        if (i + rad[i] > right) {
            right = i + rad[i];
            center = i;
        }
    }
    return t;
}

int main() {
    int len;
    string X, Y;
    cin >> len >> X >> Y;

    vector<int> rx, ry;
    string tx = buildString(X, rx);
    string ty = buildString(Y, ry);

    int sz = (int)tx.size();

    vector<ULL> hsh(sz + 1, 0), pow(sz + 1, 1);
    for (int i = 1; i <= sz; i++) pow[i] = pow[i - 1] * SEED;
    for (int i = 0; i < sz; i++) hsh[i + 1] = hsh[i] * SEED + (unsigned char)tx[i];

    vector<ULL> rhsh(sz + 1, 0);
    for (int i = sz - 1; i >= 0; i--) {
        rhsh[i] = rhsh[i + 1] * SEED + (unsigned char)ty[i];
    }

    auto getFwd = [&](int l, int r) -> ULL {
        if (l > r) return ULLONG_MAX - 1;
        return hsh[r + 1] - hsh[l] * pow[r - l + 1];
    };

    auto getRev = [&](int l, int r) -> ULL {
        if (l > r) return ULLONG_MAX - 2;
        return rhsh[l] - rhsh[r + 1] * pow[r - l + 1];
    };

    int best = 1;

    for (int i = 2; i < sz; i++) {
        int cur = max(rx[i], ry[i - 2]);

        int L = i - cur + 1;
        int R = i - 2 + cur - 1;

        int lo = 0;
        int hi = min(L, sz - 1 - R);
        int ext = 0;

        while (lo <= hi) {
            int mid = (lo + hi) >> 1;

            int l1 = L - mid;
            int r1 = L - 1;

            int l2 = R + 1;
            int r2 = R + mid;

            if (getFwd(l1, r1) == getRev(l2, r2)) {
                ext = mid;
                lo = mid + 1;
            } else {
                hi = mid - 1;
            }
        }

        best = max(best, (cur - 1) + ext);
    }

    cout << best << '\n';
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <string>
#include <set>
#include <algorithm>

using namespace std;

const int LIMIT = 1000005;
int lastIn[LIMIT];
int lastOut[LIMIT];

void process() {
    int n;
    while (cin >> n) {
        set<int> pending;
        int maxVal = 0;
        int answer = -1;
        
        vector<pair<char, int>> cmds(n);
        for (int i = 0; i < n; ++i) {
            string token;
            cin >> token;
            if (token == "?") {
                cmds[i] = {'?', 0};
            } else {
                cin >> cmds[i].second;
                cmds[i].first = token[0];
                maxVal = max(maxVal, cmds[i].second);
            }
        }

        fill(lastIn, lastIn + maxVal + 1, -1);
        fill(lastOut, lastOut + maxVal + 1, -1);

        for (int i = 0; i < n; ++i) {
            char type = cmds[i].first;
            int idx = cmds[i].second;

            if (type == '?') {
                pending.insert(i);
            } else if (type == 'I') {
                if (lastIn[idx] > lastOut[idx]) {
                    auto it = pending.lower_bound(lastIn[idx]);
                    if (it != pending.end() && *it < i) {
                        lastOut[idx] = *it;
                        pending.erase(it);
                    } else {
                        answer = i + 1;
                        break;
                    }
                }
                lastIn[idx] = i;
            } else if (type == 'O') {
                if (lastOut[idx] > lastIn[idx] || (lastOut[idx] == -1 && lastIn[idx] == -1)) {
                    int start = max(lastOut[idx], -1);
                    auto it = pending.lower_bound(start);
                    if (it != pending.end() && *it < i) {
                        lastIn[idx] = *it;
                        pending.erase(it);
                    } else {
                        answer = i + 1;
                        break;
                    }
                }
                lastOut[idx] = i;
            }
        }
        
        cout << answer << "\n";
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    process();
    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <numeric>

using namespace std;

struct UF {
    vector<int> par;
    UF(int sz) {
        par.resize(sz);
        iota(par.begin(), par.end(), 0);
    }
    int get(int x) {
        if (par[x] == x)
            return x;
        return par[x] = get(par[x]);
    }
    bool merge(int x, int y) {
        int rx = get(x);
        int ry = get(y);
        if (rx != ry) {
            par[rx] = ry;
            return true;
        }
        return false;
    }
};

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    int cnt;
    if (!(cin >> cnt)) return 0;
    vector<pair<int, int>> arr(cnt);
    for (int i = 0; i < cnt; ++i) {
        cin >> arr[i].first >> arr[i].second;
    }

    UF uf(cnt);
    int comps = cnt;
    for (int i = 0; i < cnt; ++i) {
        for (int j = i + 1; j < cnt; ++j) {
            if (arr[i].first == arr[j].first || arr[i].second == arr[j].second) {
                if (uf.merge(i, j)) {
                    comps--;
                }
            }
        }
    }

    cout << comps - 1 << "\n";
    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

using ULL = unsigned long long;
static const ULL SEED = 1315423911ULL;

struct Hash {
    vector<ULL> pre, pow;

    void build(const string& str) {
        int sz = (int)str.size();
        pre.assign(sz + 1, 0);
        pow.assign(sz + 1, 1);
        for (int i = 1; i <= sz; ++i) {
            pre[i] = pre[i - 1] * SEED + (ULL)(str[i - 1] - 'a' + 1);
            pow[i] = pow[i - 1] * SEED;
        }
    }

    ULL get(int l, int r) const {
        if (l > r) 
            return 0;
        return pre[r] - pre[l - 1] * pow[r - l + 1];
    }
};

ULL encode(const string& str) {
    ULL val = 0;
    for (char c : str) val = val * SEED + (ULL)(c - 'a' + 1);
    return val;
}

int main() {
    string pat;
    int tot;
    cin >> pat >> tot;

    string compressed;
    compressed.reserve(pat.size());
    for (char c : pat) {
        if (c == '*' && !compressed.empty() && compressed.back() == '*') 
            continue;
        compressed.push_back(c);
    }

    vector<string> frag;
    vector<int> flag;

    string cur;
    int curFlag = 0;
    for (char c : compressed) {
        if (c == '?' || c == '*') {
            frag.push_back(cur);
            flag.push_back(curFlag);
            cur.clear();
            curFlag = (c == '?') ? 1 : 2;
        } 
        else {
            cur.push_back(c);
        }
    }
    frag.push_back(cur);
    flag.push_back(curFlag);

    int cnt = (int)frag.size();
    vector<int> sz(cnt);
    vector<ULL> code(cnt);
    for (int i = 0; i < cnt; ++i) {
        sz[i] = (int)frag[i].size();
        code[i] = encode(frag[i]);
    }

    while (tot--) {
        string str;
        cin >> str;
        int len = (int)str.size();

        Hash hsh;
        hsh.build(str);

        vector<char> dp(len + 1, 0), nxt(len + 1, 0), pref(len + 1, 0);
        dp[0] = 1;

        for (int i = 0; i < cnt; ++i) {
            fill(nxt.begin(), nxt.end(), 0);

            if (flag[i] == 2) {
                pref[0] = dp[0];
                for (int j = 1; j <= len; ++j) {
                    pref[j] = pref[j - 1] | dp[j];
                }
            }

            if (sz[i] == 0) {
                if (flag[i] == 0) {
                    for (int j = 0; j <= len; ++j) nxt[j] = dp[j];
                } 
                else if (flag[i] == 1) {
                    for (int j = 1; j <= len; ++j) nxt[j] = dp[j - 1];
                } 
                else {
                    for (int j = 0; j <= len; ++j) nxt[j] = pref[j];
                }
            } else {
                for (int j = sz[i]; j <= len; ++j) {
                    if (hsh.get(j - sz[i] + 1, j) != code[i]) 
                        continue;

                    if (flag[i] == 0) {
                        nxt[j] = dp[j - sz[i]];
                    } 
                    else if (flag[i] == 1) {
                        if (j >= sz[i] + 1) {
                            nxt[j] = dp[j - sz[i] - 1];
                        }
                    } 
                    else {
                        nxt[j] = pref[j - sz[i]];
                    }
                }
            }
            
            dp.swap(nxt);
        }
        
        cout << (dp[len] ? "YES" : "NO") << '\n';
    }
    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <iostream>
#include <string>

using namespace std;

long long dp[3][31];
int nxt[3][31];
string order[6];

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int cnt;
    cin >> cnt;
    for (int i = 0; i < 6; i++) cin >> order[i];

    for (int i = 0; i < 3; i++) {
        for (int j = 0; j < 6; j++) {
            if (order[j][0] - 'A' == i) {
                dp[i][1] = 1;
                nxt[i][1] = order[j][1] - 'A';
                break;
            }
        }
    }

    for (int k = 2; k <= cnt; k++) {
        for (int i = 0; i < 3; i++) {
            int mid = nxt[i][k - 1];
            int last = 3 - i - mid;
            int target = nxt[mid][k - 1];

            if (target == last) {
                dp[i][k] = dp[i][k - 1] + 1 + dp[mid][k - 1];
                nxt[i][k] = last;
            } else {
                dp[i][k] = dp[i][k - 1] + 1 + dp[mid][k - 1] + 1 + dp[i][k - 1];
                nxt[i][k] = mid;
            }
        }
    }

    cout << dp[0][cnt] << "\n";
    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <iostream>
#include <cmath>
#include <algorithm>

using namespace std;

int main() { 
    long long a, b, c, d;
    if (cin >> a >> b >> c >> d) {
        long long x = abs(a - c);
        long long y = abs(b - d);
        
        if (x < y) {
            swap(x, y);
        }
        
        if (x == 1 && y == 0) {
            cout << 3 << "\n";
            return 0;
        }
        if (x == 2 && y == 2) {
            cout << 4 << "\n";
            return 0;
        }
        
        long long p = (x + 1) / 2;
        long long q = (x + y + 2) / 3;
        long long ans = max(p, q);
        
        if ((ans % 2) != ((x + y) % 2)) {
            ans++;
        }
        
        cout << ans << "\n";
    }
    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        mono = []
        best = 0
         
        heights.append(0)
         
        for idx in range(len(heights)):
            while mono and heights[idx] < heights[mono[-1]]:
                cur_h = heights[mono.pop()]
                 
                cur_w = idx if not mono else idx - mono[-1] - 1
                 
                best = max(best, cur_h * cur_w)
             
            mono.append(idx)
             
        return best

## J 消防局的设立

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

const int MAXV = 1e9;

int main() {
    int cnt;
    cin >> cnt;

    vector<vector<int>> adj(cnt + 1);
    for (int i = 2; i <= cnt; ++i) {
        int par;
        cin >> par;
        adj[par].push_back(i);
    }

    vector<int> seq;
    seq.reserve(cnt);
    stack<int> stk;
    stk.push(1);
    while (!stk.empty()) {
        int cur = stk.top();
        stk.pop();
        seq.push_back(cur);
        for (int nxt : adj[cur]) stk.push(nxt);
    }
    reverse(seq.begin(), seq.end());

    using State = array<array<int, 4>, 4>;
    vector<State> best(cnt + 1);

    for (int u : seq) {
        State now;
        for (auto &row : now) 
            row.fill(MAXV);

        now[3][2] = 0;
        now[0][3] = 1;

        for (int v : adj[u]) {
            vector<tuple<int, int, int>> cand;

            for (int dv = 0; dv <= 3; ++dv) {
                for (int rv = 0; rv <= 3; ++rv) {
                    int val = best[v][dv][rv];
                    if (val >= MAXV) 
                        continue;

                    if (rv == 0) 
                        continue;

                    int nd = (dv == 3 ? 3 : min(3, dv + 1));
                    int nr = (rv == 3 ? 3 : rv - 1);

                    cand.push_back({nd, nr, val});
                }
            }

            State nxt;
            for (auto &row : nxt) row.fill(MAXV);

            for (int d1 = 0; d1 <= 3; ++d1) {
                for (int r1 = 0; r1 <= 3; ++r1) {
                    if (now[d1][r1] >= MAXV) 
                        continue;

                    for (auto [d2, r2, cost] : cand) {
                        int nd = min(d1, d2);
                        int nr = 3;

                        if (r1 != 3 && d2 > r1) 
                            nr = min(nr, r1);
                        if (r2 != 3 && d1 > r2) 
                            nr = min(nr, r2);

                        nxt[nd][nr] = min(nxt[nd][nr], now[d1][r1] + cost);
                    }
                }
            }

            now = nxt;
        }

        best[u] = now;
    }

    int res = MAXV;
    for (int d = 0; d <= 3; ++d) {
        res = min(res, best[1][d][3]);
    }

    cout << res << '\n';
    return 0;
}